# Fine-tuning NLLB-200 — Sar (Sara) → Français

### Avant de commencer
1. **GPU** : Exécution → Modifier le type d'exécution → T4 GPU
2. **Données** : uploader `sar_fr_train.jsonl` et `sar_fr_val.jsonl` dans `Mon Drive/saar_data/`

### Comment exécuter
- **Ne pas cliquer "Exécuter tout"**
- Exécuter cellule par cellule
- Après la cellule d'installation : **Runtime → Redémarrer la session** puis continuer

---
## CELLULE 1 — Installation
### ⚠️ Après cette cellule : Runtime → Redémarrer la session → puis continuer à la cellule 2

In [ ]:
# Versions fixées pour Colab Python 3.12 — évite les conflits torchvision/numpy/peft
!pip install -q \
    "numpy>=1.26,<2.0" \
    "transformers==4.46.2" \
    "peft==0.13.2" \
    "datasets==3.2.0" \
    "accelerate==1.1.1" \
    "sentencepiece==0.2.0" \
    "sacrebleu" \
    "evaluate"
print("Installation terminée — faire Runtime > Redémarrer la session avant de continuer")

---
## CELLULE 2 — Vérification (reprendre ici après redémarrage)

In [ ]:
import numpy as np
import torch

assert np.__version__.startswith('1.'), f"numpy doit être 1.x, got {np.__version__} — refaire la cellule 1"
print(f"numpy   : {np.__version__}  OK")
print(f"torch   : {torch.__version__}")
print(f"GPU     : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU nom : {torch.cuda.get_device_name(0)}")
    print(f"VRAM    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

---
## CELLULE 3 — Monter Drive et charger les données

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import json
from pathlib import Path

DATA_DIR = Path('/content/drive/MyDrive/saar_data')
SAVE_DIR = DATA_DIR / 'nllb_sar_model'

def load_jsonl(path):
    with open(path, encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_data = load_jsonl(DATA_DIR / 'sar_fr_train.jsonl')
val_data   = load_jsonl(DATA_DIR / 'sar_fr_val.jsonl')

print(f"Train : {len(train_data):,} paires")
print(f"Val   : {len(val_data):,} paires")
print(f"Exemple SAR : {train_data[0]['sar']}")
print(f"Exemple FR  : {train_data[0]['fr']}")

---
## CELLULE 4 — Tokenizer NLLB + ajout du token sar_Latn

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = 'facebook/nllb-200-distilled-600M'
SAR_LANG   = 'sar_Latn'
FR_LANG    = 'fra_Latn'

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=False)

if SAR_LANG not in tokenizer.all_special_tokens:
    tokenizer.add_special_tokens({'additional_special_tokens': [SAR_LANG]})
    print(f"Token '{SAR_LANG}' ajouté")
else:
    print(f"Token '{SAR_LANG}' déjà présent")

SAR_LANG_ID = tokenizer.convert_tokens_to_ids(SAR_LANG)
FR_LANG_ID  = tokenizer.convert_tokens_to_ids(FR_LANG)
print(f"Vocab size  : {len(tokenizer):,}")
print(f"ID sar_Latn : {SAR_LANG_ID} | ID fra_Latn : {FR_LANG_ID}")

---
## CELLULE 5 — Charger le modèle

In [ ]:
import torch
from transformers import AutoModelForSeq2SeqLM

device = 'cuda' if torch.cuda.is_available() else 'cpu'

print("Chargement du modèle (~1 min)...")
model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
)
model.resize_token_embeddings(len(tokenizer))

# Initialiser l'embedding sar_Latn depuis haoussa (langue sub-saharienne dans NLLB)
with torch.no_grad():
    hau_id = tokenizer.convert_tokens_to_ids('hau_Latn')
    model.model.shared.weight[SAR_LANG_ID] = model.model.shared.weight[hau_id].clone()

model = model.to(device)
print(f"Modèle sur {device} | {sum(p.numel() for p in model.parameters())/1e6:.0f}M paramètres")

---
## CELLULE 6 — LoRA

In [ ]:
from peft import get_peft_model, LoraConfig, TaskType

model = get_peft_model(model, LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    r=16,
    lora_alpha=32,
    lora_dropout=0.1,
    target_modules=['q_proj', 'v_proj'],
    bias='none',
))
model.print_trainable_parameters()

---
## CELLULE 7 — Tokenisation du dataset

In [ ]:
from datasets import Dataset

MAX_LEN = 128

def tokenize_batch(examples):
    # Encoder la source (Sar)
    tokenizer.src_lang = SAR_LANG
    src_enc = tokenizer(
        examples['sar'],
        max_length=MAX_LEN,
        truncation=True,
    )

    # Encoder la cible (Français) — src_lang=FR_LANG pour que le BOS soit fra_Latn
    tokenizer.src_lang = FR_LANG
    tgt_enc = tokenizer(
        examples['fr'],
        max_length=MAX_LEN,
        truncation=True,
    )
    tokenizer.src_lang = SAR_LANG  # remettre pour le batch suivant

    src_enc['labels'] = [
        [(t if t != tokenizer.pad_token_id else -100) for t in ids]
        for ids in tgt_enc['input_ids']
    ]
    return src_enc

remove_cols = ['sar', 'fr', 'source']
train_tok = Dataset.from_list(train_data).map(
    tokenize_batch, batched=True, batch_size=128,
    remove_columns=remove_cols,
)
val_tok = Dataset.from_list(val_data).map(
    tokenize_batch, batched=True, batch_size=128,
    remove_columns=remove_cols,
)
train_tok.set_format('torch')
val_tok.set_format('torch')
print(f"Train : {len(train_tok):,} | Val : {len(val_tok):,}")

---
## CELLULE 8 — Entraînement

In [ ]:
import numpy as np
import evaluate
from transformers import Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq

metric = evaluate.load('sacrebleu')

def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple):
        preds = preds[0]
    preds  = np.where(preds  != -100, preds,  tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    decoded_preds  = tokenizer.batch_decode(preds,  skip_special_tokens=True)
    decoded_labels = [[s] for s in tokenizer.batch_decode(labels, skip_special_tokens=True)]
    return {'bleu': round(metric.compute(predictions=decoded_preds, references=decoded_labels)['score'], 2)}

# 'tokenizer' renommé 'processing_class' dans transformers >= 4.46
import transformers
trainer_kwargs = dict(
    model=model,
    args=Seq2SeqTrainingArguments(
        output_dir='/content/nllb_sar_checkpoints',
        num_train_epochs=5,
        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,
        gradient_accumulation_steps=4,
        learning_rate=5e-4,
        warmup_steps=200,
        weight_decay=0.01,
        fp16=True,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='bleu',
        predict_with_generate=True,
        generation_max_length=MAX_LEN,
        logging_steps=50,
        report_to='none',
    ),
    train_dataset=train_tok,
    eval_dataset=val_tok,
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model, padding=True),
    compute_metrics=compute_metrics,
)

tv = tuple(int(x) for x in transformers.__version__.split('.')[:2])
if tv >= (4, 46):
    trainer_kwargs['processing_class'] = tokenizer
else:
    trainer_kwargs['tokenizer'] = tokenizer

trainer = Seq2SeqTrainer(**trainer_kwargs)

print(f"transformers {transformers.__version__} — Steps/epoch : {len(train_tok) // (8 * 4)}")
print("Démarrage...")
trainer.train()

---
## CELLULE 9 — Sauvegarder sur Drive

In [ ]:
SAVE_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(str(SAVE_DIR / 'lora_weights'))
tokenizer.save_pretrained(str(SAVE_DIR / 'tokenizer'))
print(f"Sauvegardé : {SAVE_DIR}")

---
## CELLULE 10 — Test de traduction

In [ ]:
def translate(text):
    tokenizer.src_lang = SAR_LANG
    inputs = tokenizer(text, return_tensors='pt').to(device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            forced_bos_token_id=FR_LANG_ID,
            max_new_tokens=128,
            num_beams=4,
        )
    decoded = tokenizer.decode(out[0], skip_special_tokens=True)
    # Supprimer le tag de langue si présent en début de texte
    if decoded.startswith(FR_LANG):
        decoded = decoded[len(FR_LANG):].strip()
    return decoded

print("=== Tests Sar → Français ===\n")
for item in val_data[:5]:
    pred = translate(item['sar'])
    print(f"SAR     : {item['sar']}")
    print(f"PRÉDIT  : {pred}")
    print(f"ATTENDU : {item['fr']}\n")